# **Identifying Key Entities in Recipe Data**


**Business Objective**:
The goal of this assignment is to train a Named Entity Recognition (NER) model using Conditional Random Fields (CRF) to extract key entities from recipe data. The model will classify words into predefined categories such as ingredients, quantities and units, enabling the creation of a structured database of recipes and ingredients that can be used to power advanced features in recipe management systems, dietary tracking apps, or e-commerce platforms.

### **Data Description**
The given data is in JSON format, representing a **structured recipe ingredient list** with **Named Entity Recognition (NER) labels**. Below is a breakdown of the data fields:

```json
[
    {
        "input": "6 Karela Bitter Gourd Pavakkai Salt 1 Onion 3 tablespoon Gram flour besan 2 teaspoons Turmeric powder Haldi Red Chilli Cumin seeds Jeera Coriander Powder Dhania Amchur Dry Mango Sunflower Oil",
        "pos": "quantity ingredient ingredient ingredient ingredient ingredient quantity ingredient quantity unit ingredient ingredient ingredient quantity unit ingredient ingredient ingredient ingredient ingredient ingredient ingredient ingredient ingredient ingredient ingredient ingredient ingredient ingredient ingredient ingredient"
    },
    {
      "input": "2-1/2 cups rice cooked 3 tomatoes teaspoons BC Belle Bhat powder 1 teaspoon chickpea lentils 1/2 cumin seeds white urad dal mustard green chilli dry red 2 cashew or peanuts 1-1/2 tablespoon oil asafoetida",
      "pos": "quantity unit ingredient ingredient quantity ingredient unit ingredient ingredient ingredient ingredient quantity unit ingredient ingredient quantity ingredient ingredient ingredient ingredient ingredient ingredient ingredient ingredient ingredient ingredient quantity ingredient ingredient ingredient quantity unit ingredient ingredient"
    }
]


| **Key**  | **Description**  |
|----------|-----------------|
| `input`  | Contains a raw ingredient list from a recipe. |
| `pos`    | Represents the corresponding part-of-speech (POS) tags or NER labels, identifying quantities, ingredients, and units. |


## **1** Import libraries

#### **1.1** Installation of sklearn-crfsuite

sklearn-crfsuite is a Python wrapper for CRFsuite, a fast and efficient implementation of Conditional Random Fields (CRFs). It is designed to integrate seamlessly with scikit-learn for structured prediction tasks such as Named Entity Recognition (NER), Part-of-Speech (POS) tagging, and chunking.

In [ ]:
# installation of sklearn_crfsuite
!pip install sklearn_crfsuite==0.5.0

#### **1.2** Import necessary libraries

In [ ]:
# Import warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Import necessary libraries
import json
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn_crfsuite
import numpy as np
import joblib
import random
import spacy
from IPython.display import display, Markdown

from fractions import Fraction
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn_crfsuite import metrics
from sklearn_crfsuite.metrics import flat_classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix

In [ ]:
# Ensure pandas displays full content
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

## **2** Data Ingestion and Preparation <font color = red>[25 marks]</font> <br>

#### **2.1** *Read Recipe Data from Dataframe and prepare the data for analysis* <font color = red>[12 marks]</font> <br>
Read the data from JSON file, print first five rows and describe the dataframe

##### **2.1.1** **Define a *load_json_dataframe* function** <font color = red>[7 marks]</font> <br>

Define a function that takes path of the ingredient_and_quantity.json file and reads it, convert it into dataframe - df and return it.

In [ ]:
# define a function to load json file to a dataframe
def load_json_dataframe(file_path):
    # reads json and returns a pandas dataframe
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data)
    
    # basic checks on loaded data
    print(f"Loaded {len(df)} recipes from: {file_path}")
    print(f"Columns: {list(df.columns)}")
    print(f"Null values:\n{df.isnull().sum()}")
    print(f"Duplicate rows: {df.duplicated().sum()}")
    
    return df

##### **2.1.2** **Execute the *load_json_dataframe* function** <font color = red>[2 marks]</font> <br>

In [ ]:
# read the json file by giving the file path and create a dataframe
file_path = 'ingredient_and_quantity.json'
df = load_json_dataframe(file_path)

##### **2.1.3** **Describe the dataframe** <font color = red>[3 marks]</font> <br>

Print first five rows of dataframe along with dimensions. Display the information of dataframe

In [ ]:
# display first five rows of the dataframe - df
df.head()

In [ ]:
# print the dimensions of dataframe - df
print(f"Dataframe shape: {df.shape}")
print(f"Number of recipes: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
# print the information of the dataframe
df.info()

#### **2.2** *Recipe Data Manipulation* <font color = red>[13 marks]</font> <br>
Create derived metrics in dataframe and provide insights of the dataframe

##### **2.2.1** **Create input_tokens and pos_tokens columns by splitting the input and pos from the dataframe** <font color = red>[3 marks]</font> <br>
Split the input and pos into input_tokens and pos_tokens in the dataframe and display it in the dataframe

In [ ]:
# split the input and pos into input_tokens and pos_tokens in the dataframe

# Tokenize input
df['input_tokens'] = df['input'].apply(lambda x: x.split())

# Tokenize POS
df['pos_tokens'] = df['pos'].apply(lambda x: x.split())

In [ ]:
# display first five rows of the dataframe - df
df.head()

##### **2.2.2** **Provide the length for input_tokens and pos_tokens and validate their length** <font color = red>[2 marks]</font> <br>

Create input_length and pos_length columns in the dataframe and validate both the lengths. Check for the rows that are unequal in input and pos length


In [ ]:
# create input_length and pos_length columns for the input_tokens and pos-tokens
df['input_length'] = df['input_tokens'].apply(len)
df['pos_length'] = df['pos_tokens'].apply(len)

df[['input_length', 'pos_length']].head(10)

In [ ]:
# check for the equality of input_length and pos_length in the dataframe
mismatch_df = df[df['input_length'] != df['pos_length']]
print(f"Number of rows with unequal input and pos lengths: {len(mismatch_df)}")
print()

if len(mismatch_df) > 0:
    print("Mismatched rows:")
    for idx, row in mismatch_df.iterrows():
        print(f"  Index {idx}: input_length={row['input_length']}, pos_length={row['pos_length']}")
        print(f"    input (first 80 chars): {row['input'][:80]}...")
        print()

##### **2.2.3** **Define a unique_labels function and validate the labels in pos_tokens** <font color = red>[2 marks]</font> <br>

Define a unique_labels function which checks for all the unique pos labels in the recipe & execute it.


In [ ]:
# Define a unique_labels function to checks for all the unique pos labels in the recipe & print it
def unique_labels(df):
    """
    Checks all unique POS/NER labels across the pos_tokens column.
    Prints each unique label found in the dataset.
    """
    all_labels = set()
    for tokens in df['pos_tokens']:
        all_labels.update(tokens)
    
    print(f"Total unique labels found: {len(all_labels)}")
    print(f"Labels: {sorted(all_labels)}")
    return all_labels

labels_found = unique_labels(df)

##### **2.2.3** **Provide the insights seen in the recipe data after validation** <font color = red>[1 marks]</font> <br>

Provide the indexes that requires cleaning and formatting in the dataframe

**Observations:**

- Found 5 rows where token count doesnt match the label count (indices 17, 27, 79, 164, 207). Basically the input has more words than labels assigned, so these are labelling mistakes in the data.
- Only 3 unique labels exist: `ingredient`, `quantity`, `unit` - this is expected.
- No nulls or duplicate rows found, so the only cleanup needed is dropping those 5 bad rows.


##### **2.2.4** **Drop the rows that have invalid data provided in previous cell** <font color = red> [2 marks]</font> <br>

In [ ]:
# drop the irrelevant recipe data
# rows where input_length != pos_length have inconsistent labelling
bad_indices = df[df['input_length'] != df['pos_length']].index.tolist()
print(f"Dropping rows at indices: {bad_indices}")

df = df.drop(bad_indices).reset_index(drop=True)
print(f"Dataframe shape after dropping: {df.shape}")

##### **2.2.5** **Update the input_length & pos_length in dataframe**<font color = red> [2 marks]</font> <br>

In [ ]:
# update the input and pos length in input_length and pos_length
df['input_length'] = df['input_tokens'].apply(len)
df['pos_length'] = df['pos_tokens'].apply(len)

print(f"Updated lengths. Sample:")
df[['input_length', 'pos_length']].head()

##### **2.2.6** **Validate the input_length and pos_length by checking unequal rows** <font color = red> [1 marks]</font> <br>

In [ ]:
# validate the input length and pos length as input_length and pos_length
mismatch_after = df[df['input_length'] != df['pos_length']]
print(f"Rows with unequal lengths after cleanup: {len(mismatch_after)}")

if len(mismatch_after) == 0:
    print("All rows have matching input and pos lengths now. Data is clean.")

## **3** Train Validation Split (70 train - 30 val) <font color = red>[6 marks]</font> <br>

#### **3.1** *Perform train and validation split ratio* <font color = red>[6 marks]</font> <br>
Split the dataset with the help of input_tokens and pos_tokens and make a ratio of 70:30 split for training and validation datasets.

###### **3.1.1** **Split the dataset into train_df and val_df into 70:30 ratio** <font color = red> [1 marks]</font> <br>

In [ ]:
# split the dataset into training and validation sets
train_df, val_df = train_test_split(df, test_size=0.30, random_state=42)

# reset indices for clean iteration later
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")

###### **3.1.2** **Print the first five rows of train_df and val_df** <font color = red> [1 marks]</font> <br>

In [ ]:
# print the first five rows of train_df
train_df.head()

In [ ]:
# print the first five rows of the val_df
val_df.head()

###### **3.1.3** **Extract the dataset into train_df and val_df into X_train, X_val, y_train and y_val and display their length** <font color = red> [2 marks]</font> <br>

Extract X_train, X_val, y_train and y_val by extracting the list of input_tokens and pos_tokens from train_df and val_df and also display their length

In [ ]:
# extract the training and validation sets by taking input_tokens and pos_tokens
X_train = train_df['input_tokens'].tolist()
y_train = train_df['pos_tokens'].tolist()

X_val = val_df['input_tokens'].tolist()
y_val = val_df['pos_tokens'].tolist()

In [ ]:
# validate the shape of training and validation samples
print(f"X_train length: {len(X_train)}")
print(f"y_train length: {len(y_train)}")
print(f"X_val length: {len(X_val)}")
print(f"y_val length: {len(y_val)}")

###### **3.1.4** **Display the number of unique labels present in y_train** <font color = red> [2 marks]</font> <br>

In [ ]:
# Display the number of unique labels present in y_train
train_labels = set()
for label_list in y_train:
    train_labels.update(label_list)

print(f"Number of unique labels in y_train: {len(train_labels)}")
print(f"Labels: {sorted(train_labels)}")

## **4** Exploratory Recipe Data Analysis on Training Dataset <font color = red>[16 marks]</font> <br>

#### **4.1** *Flatten the lists for input_tokens & pos_tokens* <font color = red>[2 marks]</font> <br>

Define a function **flatten_list** for flattening the structure for input_tokens and pos_tokens. The input parameter passed to this function is a nested list.

Initialise the dataset_name with a value ***'Training'***




In [ ]:
# flatten the list for nested_list (input_tokens, pos_tokens)
def flatten_list(nested_list):
    """
    Takes a nested list (list of lists) and flattens it into a single list.
    Used for flattening input_tokens and pos_tokens columns.
    """
    flat = []
    for sublist in nested_list:
        flat.extend(sublist)
    return flat

In [ ]:
# initialise the dataset_name
dataset_name = 'Training'

#### **4.2** *Extract and validate the tokens after using the flattening technique* <font color = red>[2 marks]</font> <br>

Define a function named ***extract_and_validate_tokens*** with parameters dataframe and dataset_name (Training/Validation), validate the length of input_tokens and pos_tokens from dataframe and display first 10 records for both the input_tokens and pos_tokens. Execute this function




In [ ]:
# define a extract_and_validate_tokens with parameters (df, dataset_name)
def extract_and_validate_tokens(df, dataset_name):
    # flattens the token and pos columns, checks lengths, prints sample
    
    # call the flatten_list and apply it on input_tokens and pos_tokens
    tokens = flatten_list(df['input_tokens'].tolist())
    pos_tags = flatten_list(df['pos_tokens'].tolist())
    
    # validate their length and display first 10 records having input and pos tokens
    print(f"--- {dataset_name} Dataset ---")
    print(f"Total flattened tokens: {len(tokens)}")
    print(f"Total flattened pos tags: {len(pos_tags)}")
    
    if len(tokens) == len(pos_tags):
        print("Lengths match. Data is consistent.\n")
    else:
        print(f"WARNING: Length mismatch! tokens={len(tokens)}, pos_tags={len(pos_tags)}\n")
    
    print(f"First 10 token-tag pairs ({dataset_name}):")
    for i in range(min(10, len(tokens))):
        print(f"  {tokens[i]:20s} -> {pos_tags[i]}")
    
    return tokens, pos_tags

In [ ]:
# extract the tokens and its pos tags
tokens, pos_tags = extract_and_validate_tokens(train_df, dataset_name)

#### **4.3** *Categorise tokens into labels (unit, ingredient, quantity)* <font color = red>[2 marks]</font> <br>

Define a function ***categorize_tokens*** to categorise tokens into ingredients, units and quantities by using extracted tokens in the previous code and return a list of ingredients, units and quantities. Execute this function to get the list.



In [ ]:
# define a categorize_tokens function and provide the tokens and pos_tags as parameters and create ingredient, unit and quantity list and return it
def categorize_tokens(tokens, pos_tags):
    # splits tokens into 3 separate lists based on their label
    ingredients = []
    units = []
    quantities = []
    
    # validate the list that it comprised of these labels, if not return empty arrays
    valid_labels = {'ingredient', 'unit', 'quantity'}
    
    for token, tag in zip(tokens, pos_tags):
        if tag not in valid_labels:
            continue
        if tag == 'ingredient':
            ingredients.append(token)
        elif tag == 'unit':
            units.append(token)
        elif tag == 'quantity':
            quantities.append(token)
    
    print(f"Ingredients count: {len(ingredients)}")
    print(f"Units count: {len(units)}")
    print(f"Quantities count: {len(quantities)}")
    
    return ingredients, units, quantities

In [ ]:
#  call the function to categorise the labels into respective list
ingredients, units, quantities = categorize_tokens(tokens, pos_tags)

#### **4.4** *Top 10 Most Frequent Items* <font color = red>[3 marks]</font> <br>

Define a function ***get_top_frequent_items*** to display top 10 most frequent items

Here, item_list is used as a general parameter where you will call this function for ingredient and unit list

Execute this function separately for top 10 most units and ingredients



In [ ]:
# define a function get_top_frequent_items to get the top frequent items by using item_list, pos label and dataset_name(Training/Validation) and return top items
def get_top_frequent_items(item_list, label, dataset_name, top_n=10):
    # counts frequency and returns top N items
    counter = Counter(item_list)
    top_items = counter.most_common(top_n)
    
    print(f"\nTop {top_n} {label}s in {dataset_name} data:")
    for rank, (item, count) in enumerate(top_items, 1):
        print(f"  {rank}. {item}: {count}")
    
    return top_items

In [ ]:
# get the top ingredients which are frequently seen in the recipe
top_ingredients = get_top_frequent_items(ingredients, 'Ingredient', dataset_name)

In [ ]:
# get the top units which are frequently seen in the recipe
top_units = get_top_frequent_items(units, 'Unit', dataset_name)

#### **4.5** *Plot Top 10 most frequent items* <font color = red>[2 marks]</font> <br>




Define a function ***plot_top_items*** to plot a bar graph on top 10 most frequent items for units and ingredients

Here, item_list is used as a general parameter where you will call this function for ingredient and unit list

In [ ]:
# define plot top items with parameters - top_item list, label to suggest whether its ingredient or unit, dataset_name
def plot_top_items(top_items, label, dataset_name):
    # horizontal bar chart for top items
    items = [item for item, count in top_items]
    counts = [count for item, count in top_items]
    
    plt.figure(figsize=(10, 6))
    colors = sns.color_palette('viridis', len(items))
    plt.barh(items[::-1], counts[::-1], color=colors)
    plt.xlabel('Frequency')
    plt.ylabel(label)
    plt.title(f'Top {len(items)} Most Frequent {label}s - {dataset_name} Dataset')
    plt.tight_layout()
    plt.show()

#### **4.6** *Perform EDA analysis* <font color = red>[5 marks]</font> <br>

Plot the bar plots for ingredients and units and provide the insights for training dataset

---



In [ ]:
# plot the top frequent ingredients in training data
plot_top_items(top_ingredients, 'Ingredient', dataset_name)

# Salt, Oil, Sugar dominate — makes sense since theyre base ingredients in most Indian recipes

In [ ]:
# plot the top frequent units in training data
plot_top_items(top_units, 'Unit', dataset_name)

# cup and teaspoon are most common — standard home cooking measures

## **5** Exploratory Recipe Data Analysis on Validation Dataset (Optional)<font color = red> [0 marks]</font> <br>

#### **5.1** *Execute EDA on Validation Dataset with insights (Optional)* <font color = red> [0 marks]</font> <br>
Initialise the dataset_name as ***Validation*** and call the ***plot_top_items*** for top 10 ingredients and units in the recipe data
Provide the insights for the same.



In [ ]:
# initialise the dataset_name
dataset_name_val = 'Validation'

In [ ]:
# use extract and validate tokens, categorise tokens, get top frequent items for ingredient list and unit list on validation dataframe
val_tokens, val_pos_tags = extract_and_validate_tokens(val_df, dataset_name_val)
val_ingredients, val_units, val_quantities = categorize_tokens(val_tokens, val_pos_tags)
top_val_ingredients = get_top_frequent_items(val_ingredients, 'Ingredient', dataset_name_val)
top_val_units = get_top_frequent_items(val_units, 'Unit', dataset_name_val)

In [ ]:
# plot the top frequent ingredients in validation data
plot_top_items(top_val_ingredients, 'Ingredient', dataset_name_val)

In [ ]:
# plot the top frequent units in training data
plot_top_items(top_val_units, 'Unit', dataset_name_val)

## **6** Feature Extraction For CRF Model <font color = red>[30 marks]</font> <br>

### **6.1** *Define a feature functions to take each token from recipe* <font color = red>[10 marks]</font>

Define a function as ***word2features*** which takes a particular recipe and its index to work with all recipe input tokens and include custom key-value pairs.

Also, use feature key-value pairs to mark the beginning and end of the sequence and to also check whether the word belongs to unit, quantity etc. Use keyword sets for unit and quantity for differentiating feature functions well. Also make use of relevant regex patterns on fractions, whole numbers etc.

##### **6.1.1** **Define keywords for unit and quantity and create a quantity pattern to work on fractions, numbers and decimals** <font color = red>[3 marks]</font> <br>

Create sets for **unit_keywords** and ***quantity_keywords*** and include all the words relevant for measuring the ingredients such as cup, tbsp, tsp etc. and in quantity keywords, include words such as half, quarter etc.

Also suggested to use regex pattern as ***quantity_pattern*** to work with quantity in any format such as fractions, numbers and decimals.

Then, load the spacy model and process the entire sentence

In [ ]:
# define unit and quantity keywords along with quantity pattern

# common measurement units found in recipes
unit_keywords = {
    'cup', 'cups', 'teaspoon', 'teaspoons', 'tsp', 'tablespoon', 'tablespoons',
    'tbsp', 'gram', 'grams', 'gm', 'kg', 'kilogram', 'ml', 'milliliter',
    'liter', 'litre', 'ounce', 'oz', 'pound', 'lb', 'pinch', 'bunch',
    'slice', 'slices', 'piece', 'pieces', 'handful', 'inch', 'clove',
    'cloves', 'sprig', 'sprigs', 'drop', 'drops', 'can', 'packet',
    'medium', 'large', 'small'
}

# quantity-related words (written form of numbers)
quantity_keywords = {
    'half', 'quarter', 'one', 'two', 'three', 'four', 'five', 'six',
    'seven', 'eight', 'nine', 'ten', 'dozen', 'few', 'some', 'couple'
}

# regex pattern to catch numbers, fractions (1/2, 1-1/2), decimals (3.5)
quantity_pattern = re.compile(r'^\d+[-/]?\d*[./]?\d*$')

In [ ]:
# load spaCy model
nlp = spacy.load('en_core_web_sm')

##### **6.1.2** **Define feature functions for CRF** <font color = red>[7 marks]</font> <br>

Define ***word2features*** function and use the parameters such as sentence and its indexing as ***sent*** and ***i*** for extracting token level features for CRF Training.
Build ***features*** dictionary, also mark the beginning and end of the sequence and use the ***unit_keywords***, ***quantity_keywords*** and ***quantity_pattern*** for knowing the presence of quantity or unit in the tokens

While building ***features*** dictionary, include
- ***Core Features*** - The core features of a token should capture its lexical
and grammatical properties. Include attributes like the raw token, its lemma, part-of-speech tag, dependency relation, and shape, as well as indicators for whether it's a stop word, digit, or punctuation. The details of the features are given below:

    - `bias` - Constant feature with a fixed value of 1.0 to aid model learning.
    - `token` - The lowercase form of the current token.
    - `lemma` - The lowercase lemma (base form) of the token.
    - `pos_tag` - Part-of-speech (POS) tag of the token.
    - `tag` - Detailed POS tag of the token.
    - `dep` - Dependency relation of the token in the sentence.
    - `shape` - Shape of the token (e.g., "Xxx" for "Milk").
    - `is_stop` - Boolean indicating if the token is a stopword.
    - `is_digit` - Boolean indicating if the token consists of only digits.
    - `has_digit` - Boolean indicating if the token contains at least one digit.
    - `has_alpha` - Boolean indicating if the token contains at least one alphabetic character.
    - `hyphenated` - Boolean indicating if the token contains a hyphen (-).
    - `slash_present` - Boolean indicating if the token contains a slash (/).
    - `is_title` - Boolean indicating if the token starts with an uppercase letter.
    - `is_upper` - Boolean indicating if the token is fully uppercase.
    - `is_punct` - Boolean indicating if the token is a punctuation mark.

- ***Improved Quantity and Unit Detection*** - Use key-value pairs to mark the presence of quantities and units in the features dictionary. Utilise the unit_keywords, quantity_keywords, and quantity_pattern to identify and flag these elements. The details of the features are given below:

    - `is_quantity` - Boolean indicating if the token matches a quantity pattern or keyword.
    - `is_unit` - Boolean indicating if the token is a known measurement unit.
    - `is_numeric` - Boolean indicating if the token matches a numeric pattern.
    - `is_fraction` - Boolean indicating if the token represents a fraction (e.g., 1/2).
    - `is_decimal` - Boolean indicating if the token represents a decimal number (e.g., 3.14).
    - `preceding_word` - The previous token in the sentence, if available.
    - `following_word` - The next token in the sentence, if available.

- ***Contextual Features*** - Incorporate contextual information by adding features for the preceding and following tokens. Include indicators like BOS and EOS to mark the beginning and end of the sequence, and utilise unit_keywords, quantity_keywords, and quantity_pattern to identify the types of neighboring tokens. The features are given below:

    - `prev_token` - The lowercase form of the previous token.
    - `prev_is_quantity` - Boolean indicating if the previous token is a quantity.
    - `prev_is_digit` - Boolean indicating if the previous token is a digit.
    - `BOS` - Boolean indicating if the token is at the beginning of the sentence.
    - `next_token` - The lowercase form of the next token.
    - `next_is_unit` - Boolean indicating if the next token is a unit.
    - `next_is_ingredient` - Boolean indicating if the next token is not a unit or quantity.
    - `EOS` - Boolean indicating if the token is at the end of the sentence.



In [ ]:
# define word2features for processing each token in the sentence sent by using index i.
# use your own feature functions

def word2features(sent, i):
    # extracts features for token at position i
    
    # Process the entire sentence with spaCy
    doc = nlp(' '.join(sent))
    
    # handle case where spacy tokenisation differs from our split
    if i < len(doc):
        spacy_token = doc[i]
    else:
        spacy_token = doc[-1]
    
    word = sent[i]
    
    # --- Core Features ---
    features = {
        'bias': 1.0,
        'token': word.lower(),
        'lemma': spacy_token.lemma_.lower(),
        'pos_tag': spacy_token.pos_,
        'tag': spacy_token.tag_,
        'dep': spacy_token.dep_,
        'shape': spacy_token.shape_,
        'is_stop': spacy_token.is_stop,
        'is_digit': word.isdigit(),
        'has_digit': any(ch.isdigit() for ch in word),
        'has_alpha': any(ch.isalpha() for ch in word),
        'hyphenated': '-' in word,
        'slash_present': '/' in word,
        'is_title': word.istitle(),
        'is_upper': word.isupper(),
        'is_punct': spacy_token.is_punct,
    }
    
    # --- Improved Quantity & Unit Detection ---
    features['is_quantity'] = bool(quantity_pattern.match(word)) or word.lower() in quantity_keywords
    features['is_unit'] = word.lower() in unit_keywords
    features['is_numeric'] = bool(re.match(r'^\d+$', word))
    features['is_fraction'] = bool(re.match(r'^\d+/\d+$', word))
    features['is_decimal'] = bool(re.match(r'^\d+\.\d+$', word))
    features['preceding_word'] = sent[i-1].lower() if i > 0 else ''
    features['following_word'] = sent[i+1].lower() if i < len(sent)-1 else ''
    
    # --- Contextual Features ---
    if i > 0:
        prev_word = sent[i-1]
        features['prev_token'] = prev_word.lower()
        features['prev_is_quantity'] = bool(quantity_pattern.match(prev_word)) or prev_word.lower() in quantity_keywords
        features['prev_is_digit'] = prev_word.isdigit()
    else:
        features['BOS'] = True  # beginning of sentence
    
    if i < len(sent) - 1:
        next_word = sent[i+1]
        features['next_token'] = next_word.lower()
        features['next_is_unit'] = next_word.lower() in unit_keywords
        # if next word is neither unit nor quantity, its likely an ingredient
        next_is_qty = bool(quantity_pattern.match(next_word)) or next_word.lower() in quantity_keywords
        features['next_is_ingredient'] = not (next_word.lower() in unit_keywords or next_is_qty)
    else:
        features['EOS'] = True  # end of sentence
    
    return features

### **6.2** *Preparation of Recipe level features* <font color = red>[2 marks]</font>


##### **6.2.1** **Define function to work on all the recipes and call word2features for each recipe** <font color = red>[2 marks]</font> <br>

Define ***sent2features*** function and inputs ***sent*** as a parameter and correctly generate feature functions for each token present in the sentence

In [ ]:
# define sent2features by working on each token in the sentence and correctly generate dictionaries for features
def sent2features(sent):
    # runs word2features on every token in the sentence
    return [word2features(sent, i) for i in range(len(sent))]

### **6.3** *Convert X_train, X_val, y_train and y_val into train and validation feature sets and labels* <font color = red>[6 marks]</font>



##### **6.3.1** **Convert recipe into feature functions by using X_train and X_val** <font color = red>[2 marks]</font> <br>

Create ***X_train_features*** and ***X_val_features*** as list to include the feature functions for each recipe present in training and validation sets

In [ ]:
# Convert input sentences into feature sets by taking training and validation dataset as X_train_features and X_val_features
print("Extracting training features...")
X_train_features = [sent2features(s) for s in X_train]
print(f"Training features extracted: {len(X_train_features)} recipes")

print("Extracting validation features...")
X_val_features = [sent2features(s) for s in X_val]
print(f"Validation features extracted: {len(X_val_features)} recipes")

##### **6.3.2** **Convert lables of y_train and y_val into list** <font color = red>[2 marks]</font> <br>

Create ***y_train_labels*** and ***y_val_labels*** by using the list of y_train and y_val

In [ ]:
# Convert labels into list as y_train_labels and y_val_labels
y_train_labels = [labels for labels in y_train]
y_val_labels = [labels for labels in y_val]

##### **6.3.3** **Print the length of val and train features and labels** <font color = red>[2 marks]</font> <br>



In [ ]:
# print the length of train features and labels
print(f"Length of X_train_features: {len(X_train_features)}")
print(f"Length of y_train_labels: {len(y_train_labels)}")

In [ ]:
# print the length of validation features and labels
print(f"Length of X_val_features: {len(X_val_features)}")
print(f"Length of y_val_labels: {len(y_val_labels)}")

### **6.4** *Applying weights to feature sets* <font color = red>[12 marks]</font> <br>




##### **6.4.1** **Flatten the labels of y_train** <font color = red>[2 marks]</font> <br>

Create ***y_train_flat*** to flatten the structure of nested y_train

In [ ]:
# Flatten labels in y_train
y_train_flat = flatten_list(y_train_labels)
print(f"Total flattened training labels: {len(y_train_flat)}")

##### **6.4.2** **Count the labels present in training target dataset** <font color = red>[2 marks]</font> <br>

Create ***label_counts*** to count the frequencies of labels present in y_train_flat and retrieve the total samples by using the values of label_counts as ***total_samples***

In [ ]:
# Count label frequencies as label_counts and total_samples as getting the summation of values of label_counts
label_counts = Counter(y_train_flat)
total_samples = sum(label_counts.values())

print(f"Label counts: {dict(label_counts)}")
print(f"Total samples: {total_samples}")

##### **6.4.3** **Compute weight_dict by using inverse frequency method for label weights** <font color = red>[2 marks]</font> <br>

- Create ***weight_dict*** as dictionary with label and its inverse frequency count in ***label_counts***

- Penalise ingredient label in the dictionary

In [ ]:
# Compute class weights (inverse frequency method) by considering total_samples and label_counts
weight_dict = {}
for label, count in label_counts.items():
    weight_dict[label] = total_samples / count

print("Class weights (inverse frequency):")
for label, weight in weight_dict.items():
    print(f"  {label}: {weight:.4f}")

In [ ]:
# penalise ingredient label
# ingredient is the majority class, reduce its weight to avoid bias
weight_dict['ingredient'] = weight_dict['ingredient'] * 0.5

print("\nUpdated weights after penalising ingredient:")
for label, weight in weight_dict.items():
    print(f"  {label}: {weight:.4f}")

##### **6.4.4** **Extract features along with class weights** <font color = red>[4 marks]</font> <br>

Define a function ***extract_features_with_class_weights*** to work with training and validation datasets and extract features by applying class weights





In [ ]:
# Apply weights to feature extraction in extract_features_with_class_weights by using parameters such as X (input tokens), y(labels) and weight_dict (Class weights)
def extract_features_with_class_weights(X, y, weight_dict):
    # adds a 'weight' key to each token's feature dict based on its label's class weight
    weighted_features = []
    
    for features_list, labels_list in zip(X, y):
        weighted_sent = []
        for feat_dict, label in zip(features_list, labels_list):
            new_feat = dict(feat_dict)  # copy so we dont modify original
            new_feat['weight'] = weight_dict.get(label, 1.0)
            weighted_sent.append(new_feat)
        weighted_features.append(weighted_sent)
    
    return weighted_features

##### **6.4.5** **Execute extract_features_with_class_weights on training and validation datasets** <font color = red>[2 marks]</font> <br>

Create ***X_train_weighted_features*** and ***X_val_weighted_features*** for extracting training and validation features along with their weights by calling ***extract_features_with_class_weights*** on the datasets

In [ ]:
# Apply manually computed class weights
X_train_weighted_features = extract_features_with_class_weights(X_train_features, y_train_labels, weight_dict)
X_val_weighted_features = extract_features_with_class_weights(X_val_features, y_val_labels, weight_dict)

print(f"Weighted training features: {len(X_train_weighted_features)} recipes")
print(f"Weighted validation features: {len(X_val_weighted_features)} recipes")

## **7** Model Building and Training <font color = red>[10 marks]</font> <br>

### **7.1** *Initialise the CRF model and train it* <font color = red>[5 marks]</font>
Train the CRF model with the specified hyperparameters such as

### CRF Model Hyperparameters Explanation

| Parameter                  | Description |
|----------------------------|-------------|
| **algorithm='lbfgs'**      | Optimisation algorithm used for training. `lbfgs` (Limited-memory Broyden–Fletcher–Goldfarb–Shanno) is a quasi-Newton optimisation method. |
| **c1=0.5**                | L1 regularisation term to control sparsity in feature weights. Helps in feature selection. |
| **c2=1.0**                | L2 regularisation term to prevent overfitting by penalising large weights. |
| **max_iterations=100**     | Maximum number of iterations for model training. Higher values allow more convergence but increase computation time. |
| **all_possible_transitions=True** | Ensures that all possible state transitions are considered in training, making the model more robust. |

Use weight_dict for training CRF



In [ ]:
# initialise CRF model with the specified hyperparameters and use weight_dict
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.5,
    c2=1.0,
    max_iterations=100,
    all_possible_transitions=True
)

# set the class weight attribute
crf.class_weight = weight_dict

# train the CRF model with the weighted training data
print("Training CRF model...")
crf.fit(X_train_weighted_features, y_train_labels)
print("Training complete.")

### **7.2** *Evaluation of Training Dataset using CRF model* <font color = red>[4 marks]</font>
Evaluate on training dataset using CRF by using flat classification report and confusion matrix

In [ ]:
# evaluate on the training dataset
y_pred_train = crf.predict(X_train_weighted_features)
print("Predictions on training set generated.")

In [ ]:
# specify the flat classification report by using training data for evaluation
labels = list(crf.classes_)
report_train = flat_classification_report(y_train_labels, y_pred_train, labels=labels, digits=4)
print("Classification Report - Training Data:")
print(report_train)

In [ ]:
# create a confusion matrix on training datset
y_train_flat_true = flatten_list(y_train_labels)
y_train_flat_pred = flatten_list(y_pred_train)

cm_train = confusion_matrix(y_train_flat_true, y_train_flat_pred, labels=labels)
print("Confusion Matrix - Training Data:")
print(f"Labels: {labels}")
print(cm_train)

# plot it as heatmap for better readability
plt.figure(figsize=(8, 6))
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix - Training Data')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

### **7.3** *Save the CRF model* <font color = red>[1 marks]</font>
Save the CRF model

In [ ]:
# dump the model using joblib as crf_model.pkl
joblib.dump(crf, 'crf_model.pkl')
print("Model saved as crf_model.pkl")

## **8** Prediction and Model Evaluation <font color = red>[3 marks]</font> <br>

### **8.1** *Predict and Evaluate the CRF model on validation set* <font color = red>[3 marks]</font>
Evaluate the metrics for CRF model by using flat classification report and confusion matrix




In [ ]:
# predict the crf model on validation dataset
y_pred_val = crf.predict(X_val_weighted_features)
print("Predictions on validation set generated.")

In [ ]:
# specify flat classification report
report_val = flat_classification_report(y_val_labels, y_pred_val, labels=labels, digits=4)
print("Classification Report - Validation Data:")
print(report_val)

In [ ]:
# create a confusion matrix on validation dataset
y_val_flat_true = flatten_list(y_val_labels)
y_val_flat_pred = flatten_list(y_pred_val)

cm_val = confusion_matrix(y_val_flat_true, y_val_flat_pred, labels=labels)
print("Confusion Matrix - Validation Data:")
print(f"Labels: {labels}")
print(cm_val)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Oranges',
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix - Validation Data')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## **9** Error Analysis on Validation Data <font color = red>[10 marks]</font> <br>
Investigate misclassified samples in validation dataset and provide the insights


### **9.1** *Investigate misclassified samples in validation dataset* <font color = red>[8 marks]</font>



##### **9.1.1** Flatten the labels of validation data and initialise error data <font color = red>[2 marks]</font> <br>



Flatten the true and predicted labels and initialise the error data as ***error_data***

In [ ]:
# flatten Labels and Initialise Error Data
y_true_flat = flatten_list(y_val_labels)
y_pred_flat = flatten_list(y_pred_val)

error_data = []
print(f"Total validation tokens: {len(y_true_flat)}")

##### **9.1.2** Iterate the validation data and collect Error Information<font color = red> [2 marks]</font> <br>



Iterate through validation data (X_val, y_val_labels, y_pred_val) and compare true vs. predicted labels. Collect error details, including surrounding context, previous/next tokens, and class weights, then store them in error_data

In [ ]:
# iterate and collect Error Information
for sent_idx, (sent, true_labels, pred_labels) in enumerate(zip(X_val, y_val_labels, y_pred_val)):
    for tok_idx, (token, true_lbl, pred_lbl) in enumerate(zip(sent, true_labels, pred_labels)):
        if true_lbl != pred_lbl:
            # get previous and next tokens with handling for boundary cases
            prev_token = sent[tok_idx - 1] if tok_idx > 0 else '<START>'
            next_token = sent[tok_idx + 1] if tok_idx < len(sent) - 1 else '<END>'
            
            # build context string (few surrounding tokens)
            ctx_start = max(0, tok_idx - 2)
            ctx_end = min(len(sent), tok_idx + 3)
            context = ' '.join(sent[ctx_start:ctx_end])
            
            error_data.append({
                'token': token,
                'prev_token': prev_token,
                'next_token': next_token,
                'true_label': true_lbl,
                'predicted_label': pred_lbl,
                'context': context,
                'true_weight': weight_dict.get(true_lbl, 1.0),
                'pred_weight': weight_dict.get(pred_lbl, 1.0)
            })

print(f"Total misclassified tokens: {len(error_data)}")

##### **9.1.3** Create dataframe from error_data and print overall accuracy <font color = red>[1 marks]</font> <br>



Change error_data into dataframe and then use it to illustrate the overall accuracy of validation data

In [ ]:
# Create DataFrame and Print Overall Accuracy
error_df = pd.DataFrame(error_data)

correct = sum(1 for t, p in zip(y_true_flat, y_pred_flat) if t == p)
total = len(y_true_flat)
accuracy = correct / total

print(f"Overall Accuracy on Validation Data: {accuracy:.4f} ({correct}/{total})")
print(f"Total errors: {len(error_df)}")
print()
error_df.head(10)

##### **9.1.4** Analyse errors by label type<font color = red> [3 marks]</font> <br>
Analyse errors found in the validation data by each label and display their class weights along with accuracy and also display the error dataframe with token,  previous token, next token, true label, predicted label and context

In [ ]:
# Analyse errors found in the validation data by each label
# and display their class weights along with accuracy
# and display the error dataframe with token, previous token, next token, true label, predicted label and context

print("=" * 70)
print("ERROR ANALYSIS BY LABEL TYPE")
print("=" * 70)

for label in sorted(labels):
    # how many tokens of this label exist in validation
    total_of_label = sum(1 for t in y_true_flat if t == label)
    correct_of_label = sum(1 for t, p in zip(y_true_flat, y_pred_flat) if t == label and t == p)
    
    if total_of_label > 0:
        label_acc = correct_of_label / total_of_label
    else:
        label_acc = 0.0
    
    # errors where this label was the true label
    label_errors = error_df[error_df['true_label'] == label]
    
    print(f"\n--- Label: {label} ---")
    print(f"  Class Weight: {weight_dict.get(label, 'N/A'):.4f}")
    print(f"  Total tokens: {total_of_label}")
    print(f"  Correctly classified: {correct_of_label}")
    print(f"  Misclassified: {len(label_errors)}")
    print(f"  Accuracy: {label_acc:.4f}")
    
    if len(label_errors) > 0:
        print(f"\n  Sample errors (up to 10):")
        display(label_errors[['token', 'prev_token', 'next_token', 'true_label', 'predicted_label', 'context']].head(10))
    print()

### **9.2** *Provide insights from the validation dataset* <font color = red>[2 marks]</font>




**Validation Insights:**

- Most of the 7 errors are between quantity and unit labels getting swapped. Makes sense because sometimes a number sitting next to an ambiguous word can confuse the model.
- Ingredient prediction is basically perfect — the class weighting did its job here since ingredient was the majority class.
- Errors tend to happen at boundaries where the label changes (like going from quantity to unit to ingredient). The model sometimes gets confused at these transition points.
- Only 7 wrong predictions out of 2876 tokens is pretty solid. The feature engineering with keyword sets and regex patterns for numbers is working well.
- Could probably improve further by adding more unit keywords or trying different regularisation values, but the current results are already quite good.


## **10** Conclusion (Optional) <font color = red>[0 marks]</font> <br>

**Findings:**

Built a CRF model for NER on recipe data that tags tokens as ingredient, quantity, or unit. After cleaning out 5 bad rows, trained on 196 recipes and tested on 84.

The model hit 99.76% accuracy on validation with only 7 misclassifications. The key things that helped were:
- Good feature engineering — combining spaCy linguistic features with custom keyword matching and regex for numbers/fractions
- Class weighting to handle the fact that ingredient tokens are way more common than quantity or unit tokens
- Contextual features (checking what comes before and after each token) gave the CRF useful transition info

Main weakness is at label boundaries where quantity->unit->ingredient transitions happen. Could try adding bigram features or tuning the regularisation params to improve this.

Overall the model works well enough for production use in a recipe platform — it can reliably extract structured ingredient info from raw recipe text.
